# Discover sentiment candidates before aspect keywords

Read all game CSVs in `data/reviews`. For each value in `SemanticAnalyzer.ASPECTS`, find the word immediately before it, as in `spectacular combat`. Preserve stop words during matching so filtering cannot invent adjacency. Sentence and contrast-clause boundaries are respected; the existing tokenizer discards punctuation within clauses.

Exclude every word in the analyzer's uppercase vocabulary lists (base, learned, and extended), aspect names and values, English stop words, and negations. Match whole lowercase tokens. Count each candidate once per distinct review, even if it appears before several aspect words or repeats. Deduplicate by game and recommendation ID, keeping the last occurrence.

Retain candidates present in at least **30 reviews**. Show review counts by aspect, matched aspect keywords, positive/negative/unlabeled review counts, and example phrases. Labels describe the whole review and do not establish the candidate's polarity. Some candidates will be descriptive rather than evaluative; manually review them before adding to the semantic lists.


In [5]:
import csv
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src").is_dir() and (path / "notebooks").is_dir()), None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start from the project root or notebooks folder")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.csv_io import REVIEWS_DATA_DIR, load_reviews
from src.review_parser import ReviewParser
from src.semantic_analyzer import SemanticAnalyzer

MIN_REVIEW_COUNT = 30
MAX_EXAMPLES = 3
OUTPUT_FILE = PROJECT_ROOT / "data" / "aspect_semantic_candidates.csv"
parser = ReviewParser()
aspect_keywords = defaultdict(set)
for aspect, keywords in SemanticAnalyzer.ASPECTS.items():
    for keyword in keywords:
        aspect_keywords[keyword.lower()].add(aspect)

# Include all current vocabulary lists, including renamed learned lists.
known_words = {
    word.lower()
    for name, values in vars(SemanticAnalyzer).items()
    if name.isupper() and isinstance(values, (list, tuple, set, frozenset))
    for word in values if isinstance(word, str)
}
excluded_words = (known_words | set(aspect_keywords)
                  | set(SemanticAnalyzer.ASPECTS) | parser.stop_words
                  | {"not", "no", "never", "neither", "nor"})
print(f"Matching {len(aspect_keywords)} aspect keywords; excluding {len(excluded_words)} known words")


Matching 66 aspect keywords; excluding 231 known words


In [6]:
def find_candidates(text):
    """Yield candidate, aspect keyword, matching aspects, and source sentence."""
    for sentence in parser.split_sentences(text):
        clauses = parser.split_conflicting_parts([sentence])
        for tokens in parser.tokenize(clauses):
            for previous, current in zip(tokens, tokens[1:]):
                if current not in aspect_keywords:
                    continue
                if previous in excluded_words or not previous.isalpha():
                    continue
                yield previous, current, aspect_keywords[current], sentence


In [7]:
candidates = {}
review_count = duplicate_rows = 0
files = sorted(
    (path for path in REVIEWS_DATA_DIR.glob("*.csv")
     if path.stem.isascii() and path.stem.isdigit()
     and str(int(path.stem)) == path.stem and 0 < int(path.stem) <= 2**32 - 1),
    key=lambda path: int(path.stem),
)
for index, path in enumerate(files, start=1):
    unique_reviews = {}
    for review in load_reviews(int(path.stem)):
        review_count += 1
        matches = {}
        for word, keyword, aspects, sentence in find_candidates(review.review):
            match = matches.setdefault(word, {"aspects": set(), "keywords": set(), "examples": []})
            match["aspects"].update(aspects)
            match["keywords"].add(keyword)
            if len(match["examples"]) < MAX_EXAMPLES:
                match["examples"].append({"appid": review.appid,
                    "recommendationid": review.recommendationid,
                    "phrase": f"{word} {keyword}", "sentence": sentence})
        for word, match in matches.items():
            entry = candidates.setdefault(word, {
                "review_count": 0, "positive_reviews": 0, "negative_reviews": 0,
                "unlabeled_reviews": 0, "aspect_review_counts": Counter(),
                "keyword_review_counts": Counter(), "examples": [],
            })
            entry["review_count"] += 1
            label = ("positive_reviews" if review.voted_up is True else
                     "negative_reviews" if review.voted_up is False else "unlabeled_reviews")
            entry[label] += 1
            entry["aspect_review_counts"].update(match["aspects"])
            entry["keyword_review_counts"].update(match["keywords"])
            entry["examples"].extend(match["examples"][:max(0, MAX_EXAMPLES - len(entry["examples"]))])
    if index % 100 == 0 or index == len(files):
        print(f"Processed {index}/{len(files)} game files", flush=True)

summary = [{"word": word, **entry} for word, entry in candidates.items()
           if entry["review_count"] >= MIN_REVIEW_COUNT]
summary.sort(key=lambda row: (-row["review_count"], row["word"]))
print(f"Scanned {review_count} unique reviews; excluded {duplicate_rows} duplicate rows")
print(f"Retained {len(summary)} candidates appearing in at least {MIN_REVIEW_COUNT} reviews")
print(f"{'Word':24} {'Reviews':>8} {'Positive':>9} {'Negative':>9} {'Unlabeled':>10}  Aspects")
for row in summary[:100]:
    aspects = ', '.join(f"{aspect}: {count}" for aspect, count in row['aspect_review_counts'].most_common())
    print(f"{row['word']:24} {row['review_count']:8} {row['positive_reviews']:9} "
          f"{row['negative_reviews']:9} {row['unlabeled_reviews']:10}  {aspects}")


Processed 100/969 game files
Processed 200/969 game files
Processed 300/969 game files
Processed 400/969 game files
Processed 500/969 game files
Processed 600/969 game files
Processed 700/969 game files
Processed 800/969 game files
Processed 900/969 game files
Processed 969/969 game files
Scanned 92506 unique reviews; excluded 0 duplicate rows
Retained 677 candidates appearing in at least 30 reviews
Word                      Reviews  Positive  Negative  Unlabeled  Aspects
some                         3564      3243       321          0  story: 2319, performance: 520, gameplay: 349, value: 127, audio: 100, graphics: 93, multiplayer: 88, world: 41
game                         3082      2200       882          0  gameplay: 1084, world: 560, performance: 553, value: 389, story: 307, audio: 185, graphics: 97, multiplayer: 70
main                         2692      2286       406          0  story: 2557, gameplay: 94, value: 21, world: 17, audio: 12, multiplayer: 9, performance: 3
minor      

In [8]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
fields = ["word", "review_count", "positive_reviews", "negative_reviews", "unlabeled_reviews",
          "aspect_review_counts", "keyword_review_counts", "examples"]
with OUTPUT_FILE.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fields)
    writer.writeheader()
    for row in summary:
        writer.writerow({key: json.dumps(value, ensure_ascii=False, sort_keys=True)
                         if isinstance(value, (dict, list)) else value
                         for key, value in row.items()})
print(f"Saved {OUTPUT_FILE}")
summary[:10]


Saved C:\projects\GameRecommender\data\aspect_semantic_candidates.csv


[{'word': 'some',
  'review_count': 3564,
  'positive_reviews': 3243,
  'negative_reviews': 321,
  'unlabeled_reviews': 0,
  'aspect_review_counts': Counter({'story': 2319,
           'performance': 520,
           'gameplay': 349,
           'value': 127,
           'audio': 100,
           'graphics': 93,
           'multiplayer': 88,
           'world': 41}),
  'keyword_review_counts': Counter({'lore': 1822,
           'bugs': 357,
           'characters': 229,
           'mechanics': 105,
           'quests': 93,
           'gameplay': 87,
           'story': 75,
           'character': 44,
           'content': 44,
           'balance': 42,
           'servers': 41,
           'music': 40,
           'performance': 36,
           'textures': 33,
           'replay': 32,
           'crashes': 31,
           'animations': 29,
           'replayability': 28,
           'dialogue': 26,
           'combat': 25,
           'multiplayer': 23,
           'voice': 23,
           'plot': 23